In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
field = 'eco_bus'
file = 'DATAFILES/data_'+ field + '.xlsx'
journals = pd.read_excel(file,sheet_name='journals')
rjour = journals[['journal','acr','pub']].copy()
datalist = pd.read_pickle('HOME_PICKLE_FILES/jdatalist.pkl')

In [3]:
def matrix(datalist,  journals):
    njour = journals.copy()
    for k in range(len(datalist)):
        data = datalist[k]
        data = njour.merge(data,on='journal',how='left').dropna()
        data = data.drop(['acr', 'pub','journal_score'], axis=1)
        datacero = data.drop_duplicates(subset=['UT', 'journal'], keep='first')
        datacero = datacero[['UT', 'journal']]
        sj =  datacero.groupby('journal')['UT'].count()
        nsj = pd.DataFrame(sj)
        nsj = nsj.reset_index()
        nsj =nsj.rename(columns={'UT':k})
        nsj = journals.merge(nsj, on='journal',how='left').fillna(0)
        nsj = nsj[['acr',k]]
        njour = njour.merge(nsj,on='acr',how='left').fillna(0)
    njour = njour.drop(['journal', 'acr','pub','journal_score'], axis=1)
    matrix = njour.to_numpy()
    colsum = np.sum(matrix,axis=1)
    B = matrix / colsum[:, np.newaxis]#eigenvector method
    #A = matrix / colsum.reshape(-1,1)#PN method
    A = matrix / colsum#PN method
    A = A.T
    B = B.T
    return A, B, njour

matriz = arr = np.array([2, 2, 2, 1, 0, 1, 5,1,6])
matrix = matriz.reshape(3, 3)
colsum = np.sum(matrix,axis=1)
B = matrix / colsum[:, np.newaxis]#eigenvector method
A = matrix / colsum#PN method
A = A
B = B
print(matrix)
print(colsum)
print(colsum.reshape(-1,1))
print(colsum[:, np.newaxis])
print(A)
print(B)

In [4]:
def updatescores(journals,  matrix):### computing the first eigenvector of BA
    v = journals['journal_score'].to_numpy()
    oldnorma = 10
    norma = 1
    while np.abs(oldnorma-norma) > 0.00001:
        oldnorma = norma
        v = np.matmul(matrix,v)
        norma = np.linalg.norm(v)
        v = v/norma
    return v

In [5]:
#Matrices for PN and eigenvector
A, B, njour  = matrix(datalist, journals)
#solutions for PN and eigenvector
v = updatescores(journals,  A)### PN
vv = updatescores(journals,  B)### eigenvector

In [6]:
#PN solution
df = pd.DataFrame(v, columns=['journal_score'])
vjour = rjour.join(df)
vmax = vjour['journal_score'].max()
vjour['journal_score'] = 10*vjour['journal_score'] / vmax
vjour = vjour.sort_values(by=['journal_score'],ascending=False)
 
#eigenvector solution
df = pd.DataFrame(vv, columns=['journal_score'])
vvjour = rjour.join(df)
vvjour['journal_score'] = vvjour['journal_score']/vvjour['pub']#### SIZE-INDEPENDENT INFLUENCE PER PAPER
vvmax = vvjour['journal_score'].max()
vvjour['journal_score'] = 10*vvjour['journal_score'] / vvmax
vvjour = vvjour.sort_values(by=['journal_score'],ascending=False)

In [7]:
filename = 'RESULTS/PN_results.xlsx'
with pd.ExcelWriter(filename) as writer:
    vjour.to_excel(writer,sheet_name= 'PN_solution')
    vvjour.to_excel(writer,sheet_name='eigenvector_solution')
    njour.to_excel(writer,sheet_name= 'matrix')
    pd.DataFrame(A).to_excel(writer,sheet_name= 'A')
    pd.DataFrame(B).to_excel(writer,sheet_name='B')
 


In [8]:
print('The end')

The end
